# Week 5 (starter): RAG Pipeline with Retrieval Evaluation

Retrieval runs fully local and free; only generation needs a key (set `GEMINI_API_KEY`, else it is skipped with a notice). Cells marked **TODO (you)** are yours. Dependencies: `sentence-transformers`, `faiss-cpu`. For generation: `pip install openai`.

In [89]:
%pip install beautifulsoup4
%pip install faiss-cpu


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [90]:
from pathlib import Path
from bs4 import BeautifulSoup

documents = []

for path in Path("documents").glob("*.html"):
    html = path.read_text(encoding="utf-8")

    soup = BeautifulSoup(html, "html.parser")

    # Remove things we don't want embedded
    for tag in soup(["script", "style", "nav"]):
        tag.decompose()

    text = soup.get_text(" ", strip=True)

    documents.append({
        "title": path.stem,
        "text": text
    })

print(f"Loaded {len(documents)} documents")

for doc in documents:
    print(doc["title"], len(doc["text"]))

Loaded 9 documents
Spring-Authorization 29689
Spring-Security 32542
Spring-Data-JPA 5514
Spring-Security-OAuth2 22775
Spring-Batch 11438
Spring-Data 8778
Spring-Dependencies 24266
Bean-Annotation 17495
Spring-JPA-Repository 8778


In [91]:
import os, re, pathlib, numpy as np
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer
import faiss
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# TODO (you): replace with your own technical-doc corpus. This placeholder is one short doc.
DOC = "\n\n".join(
    f"## {doc['title']}\n{doc['text']}"
    for doc in documents
)

print('corpus chars:', len(DOC))

corpus chars: 161477


In [92]:
def chunk_fixed(text, size, overlap):
    text = re.sub(r'\s+', ' ', text).strip(); out, i = [], 0
    while i < len(text):
        out.append(text[i:i+size]); i += size - overlap
    return out
def chunk_paragraph(text):
    return [re.sub(r'\s+',' ',p).strip() for p in text.split('\n\n') if p.strip()]

configs = {'small (120/20)': chunk_fixed(DOC,120,20), 'large (320/40)': chunk_fixed(DOC,320,40), 'by-paragraph': chunk_paragraph(DOC)}
for n, c in configs.items():
    print(f'{n:16s} -> {len(c)} chunks')

def build(chunks):
    e = embedder.encode(chunks, normalize_embeddings=True).astype('float32')
    idx = faiss.IndexFlatIP(e.shape[1]); idx.add(e); return idx
def retrieve(idx, chunks, q, k=3):
    qe = embedder.encode([q], normalize_embeddings=True).astype('float32')
    # FAISS uses -1 for missing results when fewer than k chunks exist.
    return [chunks[i] for i in idx.search(qe, k)[1][0] if i >= 0]

small (120/20)   -> 1601 chunks
large (320/40)   -> 572 chunks
by-paragraph     -> 19 chunks


## Part 2: Evaluate retrieval (compare chunking configs)
A query is answered only if the retrieved context contains the answer fact. This is the model-free half of answer quality.

In [93]:
# TODO (you): your own queries and the fact each answer must contain.
qa = [
    (
        "What does the @Bean annotation do?",
        "register a bean definition"
    ),
    (
        "What do GrantedAuthority objects represent in Spring Security?",
        "authorities that have been granted to the principal"
    ),
    (
        "Is Spring Batch a scheduling framework?",
        "Spring Batch is not a scheduling framework"
    ),
    (
        "What does Spring Data JPA provide?",
        "repository support for the Jakarta Persistence API"
    ),
    (
        "What is the central interface in the Spring Data repository abstraction?",
        "Repository"
    ),
    (
        "What are the two main types of dependency injection in Spring?",
        "constructor-based and setter-based dependency injection"
    ),
    (
        "What functionality do CrudRepository and ListCrudRepository provide?",
        "CRUD functionality for the entity class being managed"
    ),
    (
        "What is the base property prefix for OAuth Client registration properties?",
        "spring.security.oauth2.client.registration"
    ),
    (
        "What does SecurityFilterChain determine in Spring Security?",
        "which Spring Security Filter instances should be invoked for the current request"
    ),
    (
    "What is the purpose of DelegatingFilterProxy in Spring Security?",
    "delegate to a Filter implementation that is defined as a Spring Bean"
    )
]

indices = {n: build(c) for n, c in configs.items()}
# This example checks whether retrieved context contains the answer.
# It does not calculate the chunk-level precision and recall required below.
for n, c in configs.items():
    hits = sum(1 for q, fact in qa if fact.lower() in ' '.join(retrieve(indices[n], c, q)).lower())
    print(f'{n:16s} answer-recall@3: {hits}/{len(qa)} = {hits/len(qa):.2f}')
# TODO (you): compare at least two embedding models OR at least three chunking
# configurations on at least ten queries. Label relevant chunks in advance,
# report chunk-level precision and recall, and explain where the choices disagree.

# for all the chunks we grabbed for small question one, did anyone of them contain the ground truth

small (120/20)   answer-recall@3: 3/10 = 0.30
large (320/40)   answer-recall@3: 5/10 = 0.50
by-paragraph     answer-recall@3: 7/10 = 0.70


In [94]:
# Does this chunk contain information that meaningfully helps answer this specific question?

# Manual relevance labels
# 1 = relevant
# 0 = not relevant
#
# For each question:
# - small has 3 labels for its top 3 chunks
# - large has 3 labels for its top 3 chunks
# - by-paragraph has 3 labels for its top 3 chunks

relevance_labels = {

    "What does the @Bean annotation do?": {
        "small (120/20)": [0, 0, 0],
        "large (320/40)": [1, 0, 0],
        "by-paragraph": [0, 0, 0]
    },

    "What do GrantedAuthority objects represent in Spring Security?": {
        "small (120/20)": [0, 0, 0],
        "large (320/40)": [0, 1, 1],
        "by-paragraph": [1, 0, 0]
    },

    "Is Spring Batch a scheduling framework?": {
        "small (120/20)": [1, 1, 0],
        "large (320/40)": [0, 1, 1],
        "by-paragraph": [1, 0, 1]
    },

    "What does Spring Data JPA provide?": {
        "small (120/20)": [0, 1, 0],
        "large (320/40)": [1, 1, 0],
        "by-paragraph": [0, 1, 1]
    },

    "What is the central interface in the Spring Data repository abstraction?": {
        "small (120/20)": [1, 1, 1],
        "large (320/40)": [1, 1, 0],
        "by-paragraph": [1, 1, 0]
    },

    "What are the two main types of dependency injection in Spring?": {
        "small (120/20)": [0, 0, 0],
        "large (320/40)": [1, 1, 1],
        "by-paragraph": [1, 0, 0]
    },

    "What functionality do CrudRepository and ListCrudRepository provide?": {
        "small (120/20)": [1, 1, 1],
        "large (320/40)": [1, 1, 0],
        "by-paragraph": [1, 1, 1]
    },

    "What is the base property prefix for OAuth Client registration properties?": {
        "small (120/20)": [1, 0, 0],
        "large (320/40)": [1, 1, 0],
        "by-paragraph": [0, 0, 0]
    },

    "What does SecurityFilterChain determine in Spring Security?": {
        "small (120/20)": [0, 1, 0],
        "large (320/40)": [0, 0, 1],
        "by-paragraph": [1, 0, 0]
    },

    "What is the purpose of DelegatingFilterProxy in Spring Security?": {
        "small (120/20)": [1, 1, 0],
        "large (320/40)": [0, 0, 1],
        "by-paragraph": [1, 0, 0]
    }
}

In [95]:
precision_results = {}

for question, configs_for_question in relevance_labels.items():
    precision_results[question] = {}

    for config_name, labels in configs_for_question.items():
        precision = sum(labels) / len(labels)
        precision_results[question][config_name] = precision

for question, results in precision_results.items():
    print(f"\n{question}")

    for config_name, precision in results.items():
        print(f"  {config_name}: Precision@3 = {precision:.2f}")


What does the @Bean annotation do?
  small (120/20): Precision@3 = 0.00
  large (320/40): Precision@3 = 0.33
  by-paragraph: Precision@3 = 0.00

What do GrantedAuthority objects represent in Spring Security?
  small (120/20): Precision@3 = 0.00
  large (320/40): Precision@3 = 0.67
  by-paragraph: Precision@3 = 0.33

Is Spring Batch a scheduling framework?
  small (120/20): Precision@3 = 0.67
  large (320/40): Precision@3 = 0.67
  by-paragraph: Precision@3 = 0.67

What does Spring Data JPA provide?
  small (120/20): Precision@3 = 0.33
  large (320/40): Precision@3 = 0.67
  by-paragraph: Precision@3 = 0.67

What is the central interface in the Spring Data repository abstraction?
  small (120/20): Precision@3 = 1.00
  large (320/40): Precision@3 = 0.67
  by-paragraph: Precision@3 = 0.67

What are the two main types of dependency injection in Spring?
  small (120/20): Precision@3 = 0.00
  large (320/40): Precision@3 = 1.00
  by-paragraph: Precision@3 = 0.33

What functionality do CrudRepo

In [96]:
config_names = [
    "small (120/20)",
    "large (320/40)",
    "by-paragraph"
]

for config_name in config_names:
    scores = [
        precision_results[question][config_name]
        for question in precision_results
    ]

    average_precision = sum(scores) / len(scores)

    print(f"{config_name}: Average Precision@3 = {average_precision:.2f}")

small (120/20): Average Precision@3 = 0.43
large (320/40): Average Precision@3 = 0.60
by-paragraph: Average Precision@3 = 0.43


In [97]:
recall_results = {}

for question, configs_for_question in relevance_labels.items():
    recall_results[question] = {}

    for config_name, labels in configs_for_question.items():
        # 1.0 if at least one relevant chunk was retrieved
        # 0.0 if none of the top 3 chunks were relevant
        recall = 1.0 if sum(labels) > 0 else 0.0

        recall_results[question][config_name] = recall


for question, results in recall_results.items():
    print(f"\n{question}")

    for config_name, recall in results.items():
        print(f"  {config_name}: Answer Recall@3 = {recall:.2f}")


What does the @Bean annotation do?
  small (120/20): Answer Recall@3 = 0.00
  large (320/40): Answer Recall@3 = 1.00
  by-paragraph: Answer Recall@3 = 0.00

What do GrantedAuthority objects represent in Spring Security?
  small (120/20): Answer Recall@3 = 0.00
  large (320/40): Answer Recall@3 = 1.00
  by-paragraph: Answer Recall@3 = 1.00

Is Spring Batch a scheduling framework?
  small (120/20): Answer Recall@3 = 1.00
  large (320/40): Answer Recall@3 = 1.00
  by-paragraph: Answer Recall@3 = 1.00

What does Spring Data JPA provide?
  small (120/20): Answer Recall@3 = 1.00
  large (320/40): Answer Recall@3 = 1.00
  by-paragraph: Answer Recall@3 = 1.00

What is the central interface in the Spring Data repository abstraction?
  small (120/20): Answer Recall@3 = 1.00
  large (320/40): Answer Recall@3 = 1.00
  by-paragraph: Answer Recall@3 = 1.00

What are the two main types of dependency injection in Spring?
  small (120/20): Answer Recall@3 = 0.00
  large (320/40): Answer Recall@3 = 1.0

In [98]:
# Display the top retrieved chunks for each query and chunking configuration
for q, fact in qa:
    print("\n" + "=" * 80)
    print("QUESTION:", q)
    print("EXPECTED FACT:", fact)

    for name, chunks in configs.items():
        print(f"\n{name}")

        # Embed the question
        qe = embedder.encode(
            [q],
            normalize_embeddings=True
        ).astype("float32")

        # Get the top 3 chunk indexes
        _, chunk_ids = indices[name].search(qe, 3)

        for rank, chunk_id in enumerate(chunk_ids[0], 1):
            if chunk_id >= 0:
                print(f"\nRank {rank} - Chunk {chunk_id}")
                print(chunks[chunk_id])


QUESTION: What does the @Bean annotation do?
EXPECTED FACT: register a bean definition

small (120/20)

Rank 1 - Chunk 1346
ean annotation in a @Configuration -annotated or in a @Component -annotated class. Declaring a Bean To declare a bean, y

Rank 2 - Chunk 1449
ultiple names, otherwise known as bean aliasing. The name attribute of the @Bean annotation accepts a String array for t

Rank 3 - Chunk 1418
fying Bean Scope Spring includes the @Scope annotation so that you can specify the scope of a bean. Using the @Scope Ann

large (320/40)

Rank 1 - Chunk 517
") is equivalent to @Bean(name = "myThing") . Bean Aliasing As discussed in Naming Beans , it is sometimes desirable to give a single bean multiple names, otherwise known as bean aliasing. The name attribute of the @Bean annotation accepts a String array for this purpose. The following example shows how to set a number

Rank 2 - Chunk 496
he @Bean annotation supports specifying arbitrary initialization and destruction callback met

In [ ]:
# andrew: add after this

In [ ]:
# Does this chunk contain information that meaningfully helps answer this specific question?

# Manual relevance labels
# 1 = relevant
# 0 = not relevant
#
# For each question:
# - small has 3 labels for its top 3 chunks
# - large has 3 labels for its top 3 chunks
# - by-paragraph has 3 labels for its top 3 chunks

relevance_labels = {

    "What does the @Bean annotation do?": {
        "small (120/20)": [0, 0, 0],
        "large (320/40)": [1, 0, 0],
        "by-paragraph": [0, 0, 0]
    },

    "What do GrantedAuthority objects represent in Spring Security?": {
        "small (120/20)": [0, 0, 0],
        "large (320/40)": [0, 1, 1],
        "by-paragraph": [1, 0, 0]
    },

    "Is Spring Batch a scheduling framework?": {
        "small (120/20)": [1, 1, 0],
        "large (320/40)": [0, 1, 1],
        "by-paragraph": [1, 0, 1]
    },

    "What does Spring Data JPA provide?": {
        "small (120/20)": [0, 1, 0],
        "large (320/40)": [1, 1, 0],
        "by-paragraph": [0, 1, 1]
    },

    "What is the central interface in the Spring Data repository abstraction?": {
        "small (120/20)": [1, 1, 1],
        "large (320/40)": [1, 1, 0],
        "by-paragraph": [1, 1, 0]
    },

    "What are the two main types of dependency injection in Spring?": {
        "small (120/20)": [0, 0, 0],
        "large (320/40)": [1, 1, 1],
        "by-paragraph": [1, 0, 0]
    },

    "What functionality do CrudRepository and ListCrudRepository provide?": {
        "small (120/20)": [1, 1, 1],
        "large (320/40)": [1, 1, 0],
        "by-paragraph": [1, 1, 1]
    },

    "What is the base property prefix for OAuth Client registration properties?": {
        "small (120/20)": [1, 0, 0],
        "large (320/40)": [1, 1, 0],
        "by-paragraph": [0, 0, 0]
    },

    "What does SecurityFilterChain determine in Spring Security?": {
        "small (120/20)": [0, 1, 0],
        "large (320/40)": [0, 0, 1],
        "by-paragraph": [1, 0, 0]
    },

    "What is the purpose of DelegatingFilterProxy in Spring Security?": {
        "small (120/20)": [1, 1, 0],
        "large (320/40)": [0, 0, 1],
        "by-paragraph": [1, 0, 0]
    }
}

## Part 3: Generate (needs a key)
Build a grounded prompt from the retrieved chunks and answer with Gemini.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

def gemini_chat(messages, model="gemini-3.6-flash", **kw):
    if not os.environ.get('GEMINI_API_KEY'):
        return None
    from openai import OpenAI
    client = OpenAI(api_key=os.environ['GEMINI_API_KEY'], base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    return client.chat.completions.create(model=model, messages=messages, **kw).choices[0].message.content

def answer(q, config='by-paragraph'):
    ctx = retrieve(indices[config], configs[config], q, k=3)
    #prompt = 'Answer only from the context.\nContext:\n- ' + '\n- '.join(ctx) + f'\nQuestion: {q}\nAnswer:'
    prompt = (
        "Answer the question using only the provided context. "
        "If the context does not contain enough information, say so.\n\n"
        "Context:\n- "
        + "\n- ".join(ctx)
        + f"\n\nQuestion: {q}\nAnswer:"
    )
    out = gemini_chat([{'role':'user','content':prompt}])
    return out if out is not None else '[API-BLOCKED] set GEMINI_API_KEY to generate; grounded prompt was built'

print(answer(
    "What is the purpose of DelegatingFilterProxy in Spring Security?",
    config="by-paragraph"
))

AQ.Ab8RN6LgcfMSOpHQxCzrO0wTMfkroE2e1gN6lQrb5ZmxZjurQw
Based on the provided context, the purpose of `DelegatingFilterProxy` is to:

1. **Bridge the Servlet container's lifecycle with Spring's `ApplicationContext`:** The Servlet container is not aware of Spring-defined Beans, so `DelegatingFilterProxy` can be registered through standard Servlet container mechanisms while delegating all work to a Spring Bean that implements `Filter`.
2. **Delay looking up Filter bean instances:** The Servlet container needs to register `Filter` instances before it starts up, but Spring loads beans later (typically via `ContextLoaderListener`). `DelegatingFilterProxy` allows delaying the lookup of the Spring-managed Filter bean until it is needed.


## Part 4 and 5: failure, submit
**TODO (you):** show a query where retrieval surfaces the wrong chunk or a chunk config splits a fact, and explain the mitigation. Then open a pull request with a result summary and review a classmate's Week 5 PR (the term's formal peer review). Use the rubric attached to the Canvas assignment for grading criteria and point values.